In [8]:
texts = [
    "i love this movie",
    "this movie is amazing",
    "what a great movie",
    "i really enjoyed this",
    "this was fantastic",
    "i hate this movie",
    "this movie is terrible",
    "what a horrible movie",
    "i really disliked this",
    "this was awful",
]

labels = [
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0
]

In [9]:
import torch 
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader

word_to_idx = {
    "<PAD>":0,
    "<UNK>":1,
}

for text in texts:
    for word in text.split():
        if word not in word_to_idx:
            word_to_idx[word] = len(word_to_idx)

vocab_size = len(word_to_idx)

print("Vocabulary Size:", vocab_size)



Vocabulary Size: 20


In [10]:
# convert setences -> int sequences

def encode(text):
    words = text.split()
    result = []

    for word in words:
        if word in word_to_idx:
            result.append(word_to_idx[word])
        else:
            result.append(word_to_idx["<UNK>"])
    return result

encoded_texts = [encode(text) for text in texts]

print('\nEncoded Texts:')
for text, encoded in zip(texts, encoded_texts):
    print(f"{text} -> {encoded}")


Encoded Texts:
i love this movie -> [2, 3, 4, 5]
this movie is amazing -> [4, 5, 6, 7]
what a great movie -> [8, 9, 10, 5]
i really enjoyed this -> [2, 11, 12, 4]
this was fantastic -> [4, 13, 14]
i hate this movie -> [2, 15, 4, 5]
this movie is terrible -> [4, 5, 6, 16]
what a horrible movie -> [8, 9, 17, 5]
i really disliked this -> [2, 11, 18, 4]
this was awful -> [4, 13, 19]


In [11]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):
        return torch.tensor(self.texts[idx], dtype = torch.long), torch.tensor(self.labels[idx], dtype = torch.float32)
    

In [13]:
class SentimentDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )


dataset = SentimentDataset(encoded_texts, labels)

In [14]:
def collate_fn(batch):

    sequences, labels = zip(*batch)

    sequences = nn.utils.rnn.pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0
    )

    labels = torch.stack(labels)

    return sequences, labels


loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

In [15]:
class SentimentRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=32):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        # (B, T)
        x = self.embedding(x)

        # (B, T, E)
        output, hidden = self.rnn(x)

        # (1, B, H) → (B, H)
        hidden = hidden.squeeze(0)

        # (B, H) → (B, 1)
        return self.fc(hidden).squeeze(1)

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentimentRNN(vocab_size).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


for epoch in range(1000):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss: {total_loss:.4f}"
        )

Epoch  10 | Loss: 0.0321
Epoch  20 | Loss: 0.0064
Epoch  30 | Loss: 0.0042
Epoch  40 | Loss: 0.0311
Epoch  50 | Loss: 0.0059
Epoch  60 | Loss: 0.0299
Epoch  70 | Loss: 0.0134
Epoch  80 | Loss: 0.0064
Epoch  90 | Loss: 0.0036
Epoch 100 | Loss: 0.0029
Epoch 110 | Loss: 0.0022
Epoch 120 | Loss: 0.0018
Epoch 130 | Loss: 0.0015
Epoch 140 | Loss: 0.0013
Epoch 150 | Loss: 0.0011
Epoch 160 | Loss: 0.0010
Epoch 170 | Loss: 0.0009
Epoch 180 | Loss: 0.0008
Epoch 190 | Loss: 0.0007
Epoch 200 | Loss: 0.0007
Epoch 210 | Loss: 0.0007
Epoch 220 | Loss: 0.0006
Epoch 230 | Loss: 0.0005
Epoch 240 | Loss: 0.0005
Epoch 250 | Loss: 0.0005
Epoch 260 | Loss: 0.0004
Epoch 270 | Loss: 0.0004
Epoch 280 | Loss: 0.0004
Epoch 290 | Loss: 0.0004
Epoch 300 | Loss: 0.0003
Epoch 310 | Loss: 0.0003
Epoch 320 | Loss: 0.0003
Epoch 330 | Loss: 0.0003
Epoch 340 | Loss: 0.0003
Epoch 350 | Loss: 0.0003
Epoch 360 | Loss: 0.0002
Epoch 370 | Loss: 0.0002
Epoch 380 | Loss: 0.0002
Epoch 390 | Loss: 0.0002
Epoch 400 | Loss: 0.0002


In [20]:
def predict(text):

    model.eval()

    encoded = encode(text)

    x = torch.tensor(
        [encoded],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        logit = model(x)

        probability = torch.sigmoid(logit).item()

    sentiment = "POSITIVE" if probability >= 0.5 else "NEGATIVE"

    print(
        f"\n{text}"
        f"\n-> {sentiment}"
        f" ({probability:.2f})"
    )


In [21]:

predict("i love this")
predict("this movie is horrible")
predict("this was amazing")
predict("i hate this")


i love this
-> POSITIVE (0.99)

this movie is horrible
-> NEGATIVE (0.00)

this was amazing
-> POSITIVE (1.00)

i hate this
-> NEGATIVE (0.00)
